# Shield SENTINEL IDS v3.0 — Full Production System
> Email Alerts | Telegram Bot | Auto IP Blocking | ML Anomaly Detection | Charts | Heatmap | PDF Report

**Runtime -> Run All**

In [ ]:
!pip install scapy flask requests scikit-learn pandas matplotlib fpdf2 --quiet
print('All dependencies installed!')

In [ ]:
import time, threading, json, csv, os, hashlib, random, ipaddress, smtplib, requests
from datetime import datetime, timedelta
from collections import defaultdict, deque
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Set
from enum import Enum
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

class ThreatLevel(Enum):
    INFO=("INFO",1); LOW=("LOW",2); MEDIUM=("MEDIUM",3)
    HIGH=("HIGH",4); CRITICAL=("CRITICAL",5)
    def __init__(self,label,priority): self.label=label; self.priority=priority

class AttackType(Enum):
    PORT_SCAN="Port Scan"; SYN_FLOOD="SYN Flood / DoS"
    HIGH_FREQ="High Frequency"; ICMP_FLOOD="ICMP Flood"
    BRUTE_FORCE="Brute Force"; NULL_SCAN="NULL Scan"
    FIN_SCAN="FIN Scan"; XMAS_SCAN="XMAS Scan"
    UDP_FLOOD="UDP Flood"; DNS_AMP="DNS Amplification"
    PAYLOAD="Payload Anomaly"; ML_ANOMALY="ML Anomaly"

PORTS={21:"FTP",22:"SSH",23:"Telnet",25:"SMTP",53:"DNS",80:"HTTP",
       110:"POP3",143:"IMAP",443:"HTTPS",3306:"MySQL",5432:"Postgres",
       6379:"Redis",27017:"MongoDB",8080:"HTTP-Alt",9200:"Elasticsearch"}

PRIVATE=[ipaddress.ip_network(n) for n in
         ["10.0.0.0/8","172.16.0.0/12","192.168.0.0/16","127.0.0.0/8"]]

GEO_HINTS={"1.":"US","5.":"Russia","8.":"China","14.":"China","24.":"US",
           "36.":"Australia","41.":"Africa","45.":"Asia-Pacific","47.":"Norway",
           "52.":"Mexico","54.":"US","58.":"China","91.":"Europe",
           "103.":"Singapore","198.":"US","203.":"Australia"}

def geo_lookup(ip):
    try:
        addr=ipaddress.ip_address(ip)
        for n in PRIVATE:
            if addr in n: return "Private/LAN"
    except: return "Invalid"
    return GEO_HINTS.get(ip.split(".")[0]+".","Unknown")

@dataclass
class Alert:
    timestamp:str; src_ip:str; dst_ip:str; src_port:int; dst_port:int
    protocol:str; attack_type:str; threat_level:str; description:str
    packet_count:int; geo_info:str="Unknown"; fingerprint:str=""
    def to_dict(self): return asdict(self)

@dataclass
class IPProfile:
    ip:str
    first_seen:float=field(default_factory=time.time)
    last_seen:float=field(default_factory=time.time)
    total_packets:int=0; total_bytes:int=0
    ports_contacted:Set[int]=field(default_factory=set)
    protocols:Set[str]=field(default_factory=set)
    alert_count:int=0; threat_score:float=0.0
    blacklisted:bool=False; whitelisted:bool=False
    packet_times:deque=field(default_factory=lambda:deque(maxlen=1000))
    syn_times:deque=field(default_factory=lambda:deque(maxlen=500))
    icmp_times:deque=field(default_factory=lambda:deque(maxlen=500))
    udp_times:deque=field(default_factory=lambda:deque(maxlen=500))
    failed_auth:Dict[int,int]=field(default_factory=dict)

class Rules:
    PORT_SCAN_T=15;   PORT_SCAN_W=10.0
    HIGH_FREQ_T=100;  HIGH_FREQ_W=5.0
    SYN_T=200;        SYN_W=5.0
    ICMP_T=50;        ICMP_W=5.0
    UDP_T=150;        UDP_W=5.0
    BRUTE_T=10;       BLACKLIST=100

class AlertLogger:
    def __init__(self,log_dir="logs"):
        os.makedirs(log_dir,exist_ok=True)
        ts=datetime.now().strftime("%Y%m%d_%H%M%S")
        self.json_path=f"{log_dir}/alerts_{ts}.json"
        self.csv_path=f"{log_dir}/alerts_{ts}.csv"
        self.suspect_path=f"{log_dir}/suspicious_ips.txt"
        self.alerts:List[Alert]=[]
        self._lock=threading.Lock()
        self._fields=["timestamp","src_ip","dst_ip","src_port","dst_port",
                      "protocol","attack_type","threat_level","description",
                      "packet_count","geo_info","fingerprint"]
        with open(self.csv_path,"w",newline="") as f:
            csv.DictWriter(f,fieldnames=self._fields).writeheader()
        print(f"Logs -> {log_dir}/")
        print(f"  JSON    : {self.json_path}")
        print(f"  CSV     : {self.csv_path}")
        print(f"  Suspects: {self.suspect_path}")
    def log(self,alert):
        with self._lock:
            self.alerts.append(alert)
            with open(self.json_path,"a") as f: f.write(json.dumps(alert.to_dict())+"\n")
            with open(self.csv_path,"a",newline="") as f:
                d={k:v for k,v in alert.to_dict().items() if k in self._fields}
                csv.DictWriter(f,fieldnames=self._fields).writerow(d)
            with open(self.suspect_path,"a") as f:
                f.write(f"{alert.timestamp}|{alert.src_ip}|{alert.attack_type}|{alert.threat_level}\n")
    def summary(self):
        counts=defaultdict(int); by_ip=defaultdict(int)
        for a in self.alerts: counts[a.attack_type]+=1; by_ip[a.src_ip]+=1
        return {"total":len(self.alerts),"by_attack":dict(counts),
                "top_ips":sorted(by_ip.items(),key=lambda x:-x[1])[:10]}

print("Core engine loaded!")


In [ ]:
# ---- CONFIGURE YOUR CREDENTIALS HERE ----
EMAIL_CONFIG = {
    "enabled": False,
    "sender": "your@gmail.com",
    "password": "your_app_password",
    "recipient": "alert@gmail.com",
    "min_level": "HIGH"
}
TELEGRAM_CONFIG = {
    "enabled": False,
    "bot_token": "YOUR_BOT_TOKEN",
    "chat_id":   "YOUR_CHAT_ID"
}
# -------------------------------------------

LEVEL_PRI = {"INFO":1,"LOW":2,"MEDIUM":3,"HIGH":4,"CRITICAL":5}

def send_email(alert):
    if not EMAIL_CONFIG["enabled"]: return
    if LEVEL_PRI.get(alert.threat_level,0) < LEVEL_PRI.get(EMAIL_CONFIG["min_level"],4): return
    try:
        msg = MIMEMultipart()
        msg["From"] = EMAIL_CONFIG["sender"]
        msg["To"]   = EMAIL_CONFIG["recipient"]
        msg["Subject"] = f"[SENTINEL] {alert.threat_level} - {alert.attack_type}"
        body = (f"SENTINEL IDS ALERT\n{'='*40}\n"
                f"Time    : {alert.timestamp}\nLevel   : {alert.threat_level}\n"
                f"Attack  : {alert.attack_type}\nSRC     : {alert.src_ip}:{alert.src_port}\n"
                f"Target  : {alert.dst_ip}:{alert.dst_port}\nGeoIP   : {alert.geo_info}\n"
                f"Details : {alert.description}\nFP      : {alert.fingerprint}\n{'='*40}")
        msg.attach(MIMEText(body,"plain"))
        with smtplib.SMTP_SSL("smtp.gmail.com",465) as s:
            s.login(EMAIL_CONFIG["sender"],EMAIL_CONFIG["password"])
            s.send_message(msg)
        print(f"  EMAIL sent: {alert.attack_type}")
    except Exception as e:
        print(f"  Email error: {e}")

def send_telegram(alert):
    if not TELEGRAM_CONFIG["enabled"]: return
    try:
        text = (f"SENTINEL ALERT\nLevel: {alert.threat_level}\n"
                f"Attack: {alert.attack_type}\nFrom: {alert.src_ip} ({alert.geo_info})\n"
                f"Target: {alert.dst_ip}:{alert.dst_port}\nInfo: {alert.description}")
        url = f"https://api.telegram.org/bot{TELEGRAM_CONFIG['bot_token']}/sendMessage"
        requests.post(url, data={"chat_id":TELEGRAM_CONFIG["chat_id"],"text":text}, timeout=5)
        print(f"  TELEGRAM sent: {alert.attack_type}")
    except Exception as e:
        print(f"  Telegram error: {e}")

def notify(alert):
    threading.Thread(target=send_email,    args=(alert,), daemon=True).start()
    threading.Thread(target=send_telegram, args=(alert,), daemon=True).start()

print("Notification system loaded!")
print("  Email   :", "ENABLED" if EMAIL_CONFIG["enabled"] else "disabled")
print("  Telegram:", "ENABLED" if TELEGRAM_CONFIG["enabled"] else "disabled")
print()
print("  To enable Email: set EMAIL_CONFIG['enabled'] = True and fill in credentials")
print("  To enable Telegram: set TELEGRAM_CONFIG['enabled'] = True, add bot token + chat_id")


In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

class MLDetector:
    def __init__(self, contamination=0.1):
        self.model   = IsolationForest(contamination=contamination, random_state=42)
        self.scaler  = StandardScaler()
        self.trained = False
        self.buffer  = []
        self.MIN_SAMPLES = 50

    def features(self, p):
        w = time.time() - 10.0
        return [
            sum(1 for t in p.packet_times if t>w),
            sum(1 for t in p.syn_times   if t>w),
            sum(1 for t in p.icmp_times  if t>w),
            sum(1 for t in p.udp_times   if t>w),
            len(p.ports_contacted),
            p.total_bytes / max(p.total_packets,1),
            p.alert_count,
            p.threat_score
        ]

    def update(self, p):
        self.buffer.append(self.features(p))
        if len(self.buffer) >= self.MIN_SAMPLES and not self.trained:
            X = self.scaler.fit_transform(self.buffer)
            self.model.fit(X)
            self.trained = True
            print(f"  [ML] Isolation Forest trained on {len(self.buffer)} samples")

    def predict(self, p):
        if not self.trained: return False
        X = self.scaler.transform([self.features(p)])
        return self.model.predict(X)[0] == -1

ml_detector = MLDetector()
print("ML Anomaly Detector ready (trains after 50 IP samples)")


In [ ]:
class ThreatDetector:
    SCORE={ThreatLevel.INFO:1,ThreatLevel.LOW:5,ThreatLevel.MEDIUM:15,
           ThreatLevel.HIGH:30,ThreatLevel.CRITICAL:50}
    ICONS={"INFO":"[i]","LOW":"[o]","MEDIUM":"[!]","HIGH":"[!!]","CRITICAL":"[SOS]"}

    def __init__(self,logger,ml=None,whitelist=None,blacklist=None):
        self.rules=Rules(); self.logger=logger; self.ml=ml
        self.profiles:Dict[str,IPProfile]={}
        self.whitelist=whitelist or set()
        self.blacklist=blacklist or set()
        self.blocked_ips:Set[str]=set()
        self._dedup:Dict[str,float]={}
        self._lock=threading.Lock()
        self.stats={"packets":0,"alerts":0,"ips":0,"bytes":0,"blocked":0,"start":time.time()}

    def _p(self,ip):
        if ip not in self.profiles:
            self.profiles[ip]=IPProfile(ip=ip); self.stats["ips"]+=1
            if ip in self.blacklist: self.profiles[ip].blacklisted=True
            if ip in self.whitelist: self.profiles[ip].whitelisted=True
        return self.profiles[ip]

    def _ok(self,key,cd=5.0):
        now=time.time()
        if key in self._dedup and now-self._dedup[key]<cd: return False
        self._dedup[key]=now; return True

    def _rate(self,times,w):
        c=time.time()-w; return sum(1 for t in times if t>c)

    def block_ip(self,ip):
        if ip not in self.blocked_ips:
            self.blocked_ips.add(ip); self.stats["blocked"]+=1
            print(f"  [AUTO-BLOCK] {ip} blocked")
            # REAL LINUX: os.system(f"iptables -A INPUT -s {ip} -j DROP")

    def _fire(self,p,dst,sp,dp,proto,atk,lvl,desc,cnt=1):
        if not self._ok(f"{p.ip}:{atk.value}"): return
        fp=hashlib.md5(f"{p.ip}{atk.value}{proto}".encode()).hexdigest()[:10]
        a=Alert(timestamp=datetime.now().isoformat(),src_ip=p.ip,dst_ip=dst,
                src_port=sp,dst_port=dp,protocol=proto,attack_type=atk.value,
                threat_level=lvl.label,description=desc,packet_count=cnt,
                geo_info=geo_lookup(p.ip),fingerprint=fp)
        self.logger.log(a); self.stats["alerts"]+=1; p.alert_count+=1
        p.threat_score+=self.SCORE[lvl]
        if p.threat_score>=self.rules.BLACKLIST:
            p.blacklisted=True; self.block_ip(p.ip)
        notify(a)
        svc=PORTS.get(dp,str(dp)); icon=self.ICONS.get(lvl.label,"[!]")
        sep="-"*60
        print(f"\n{sep}")
        print(f"{icon} [{lvl.label}] {atk.value}")
        print(f"   SRC : {p.ip}:{sp}  ->  {dst}:{dp} ({svc})")
        print(f"   GEO : {a.geo_info}   Proto:{proto}   FP:{fp}")
        print(f"   INFO: {desc}")
        print(f"   Score:{p.threat_score:.0f}/100  Blacklisted:{p.blacklisted}")
        print(sep)

    def analyze(self,pkt):
        with self._lock:
            self.stats["packets"]+=1
            src=pkt.get("src_ip","0.0.0.0"); dst=pkt.get("dst_ip","192.168.1.1")
            sp=pkt.get("src_port",0); dp=pkt.get("dst_port",80)
            proto=pkt.get("protocol","TCP"); flags=pkt.get("flags",0x002)
            size=pkt.get("size",64); payload=pkt.get("payload",b"")
            p=self._p(src)
            if p.whitelisted: return
            if p.ip in self.blocked_ips:
                print(f"  [DROP] Packet from {p.ip} blocked"); return
            p.last_seen=time.time(); p.total_packets+=1; p.total_bytes+=size
            p.packet_times.append(time.time())
            if dp: p.ports_contacted.add(dp)
            p.protocols.add(proto); self.stats["bytes"]+=size
            if self.ml: self.ml.update(p)
            self._port_scan(p,dst,dp,proto); self._high_freq(p,dst,sp,dp,proto)
            self._syn(p,dst,sp,dp,flags);    self._icmp(p,dst,proto)
            self._udp(p,dst,sp,dp,proto);    self._null(p,dst,sp,dp,flags)
            self._fin(p,dst,sp,dp,flags);    self._xmas(p,dst,sp,dp,flags)
            self._brute(p,dst,sp,dp);        self._dns(p,dst,sp,dp,size)
            self._payload(p,dst,dp,proto,payload)
            if self.ml and self.ml.trained and self.ml.predict(p):
                self._fire(p,dst,sp,dp,proto,AttackType.ML_ANOMALY,ThreatLevel.HIGH,
                    "Isolation Forest flagged abnormal traffic pattern")

    def _port_scan(self,p,dst,dp,proto):
        r=self.rules
        if len(p.ports_contacted)>=r.PORT_SCAN_T:
            rate=self._rate(p.packet_times,r.PORT_SCAN_W)
            if rate>=r.PORT_SCAN_T:
                self._fire(p,dst,0,dp,proto,AttackType.PORT_SCAN,ThreatLevel.HIGH,
                    f"Scanned {len(p.ports_contacted)} ports. Last:{PORTS.get(dp,dp)}",rate)
    def _high_freq(self,p,dst,sp,dp,proto):
        r=self.rules; rate=self._rate(p.packet_times,r.HIGH_FREQ_W)
        if rate>=r.HIGH_FREQ_T:
            self._fire(p,dst,sp,dp,proto,AttackType.HIGH_FREQ,ThreatLevel.MEDIUM,
                f"{rate} pkts in {r.HIGH_FREQ_W}s",rate)
    def _syn(self,p,dst,sp,dp,flags):
        r=self.rules
        if flags&0x002: p.syn_times.append(time.time())
        rate=self._rate(p.syn_times,r.SYN_W)
        if rate>=r.SYN_T:
            self._fire(p,dst,sp,dp,"TCP",AttackType.SYN_FLOOD,ThreatLevel.CRITICAL,
                f"{rate} SYN pkts in {r.SYN_W}s -- DoS/DDoS",rate)
    def _icmp(self,p,dst,proto):
        r=self.rules
        if proto=="ICMP": p.icmp_times.append(time.time())
        rate=self._rate(p.icmp_times,r.ICMP_W)
        if rate>=r.ICMP_T:
            self._fire(p,dst,0,0,"ICMP",AttackType.ICMP_FLOOD,ThreatLevel.HIGH,
                f"{rate} ICMP pkts in {r.ICMP_W}s",rate)
    def _udp(self,p,dst,sp,dp,proto):
        r=self.rules
        if proto=="UDP": p.udp_times.append(time.time())
        rate=self._rate(p.udp_times,r.UDP_W)
        if rate>=r.UDP_T:
            self._fire(p,dst,sp,dp,"UDP",AttackType.UDP_FLOOD,ThreatLevel.HIGH,
                f"{rate} UDP datagrams in {r.UDP_W}s",rate)
    def _null(self,p,dst,sp,dp,flags):
        if flags==0x000:
            self._fire(p,dst,sp,dp,"TCP",AttackType.NULL_SCAN,ThreatLevel.MEDIUM,
                "NULL flags (0x000) stealth probe")
    def _fin(self,p,dst,sp,dp,flags):
        if (flags&0x001) and not(flags&0x010) and not(flags&0x002):
            self._fire(p,dst,sp,dp,"TCP",AttackType.FIN_SCAN,ThreatLevel.MEDIUM,
                "FIN-only packet -- Nmap FIN stealth scan")
    def _xmas(self,p,dst,sp,dp,flags):
        if (flags&0x001)and(flags&0x008)and(flags&0x020):
            self._fire(p,dst,sp,dp,"TCP",AttackType.XMAS_SCAN,ThreatLevel.HIGH,
                "FIN+PSH+URG -- XMAS scan evasion")
    def _brute(self,p,dst,sp,dp):
        AUTH={22,23,21,25,110,143,3306,5432,6379}
        if dp in AUTH:
            p.failed_auth[dp]=p.failed_auth.get(dp,0)+1
            if p.failed_auth[dp]>=self.rules.BRUTE_T:
                self._fire(p,dst,sp,dp,"TCP",AttackType.BRUTE_FORCE,ThreatLevel.HIGH,
                    f"{p.failed_auth[dp]} attempts on {PORTS.get(dp,dp)}:{dp}",
                    p.failed_auth[dp])
    def _dns(self,p,dst,sp,dp,size):
        if dp==53 and size>512:
            self._fire(p,dst,sp,dp,"UDP",AttackType.DNS_AMP,ThreatLevel.HIGH,
                f"Oversized DNS query ({size}B) -- amplification")
    def _payload(self,p,dst,dp,proto,payload):
        if not payload: return
        SIGS=[b"\x90\x90\x90",b"/bin/sh",b"cmd.exe",b"powershell",
              b"<script>",b"UNION SELECT",b"DROP TABLE",b"eval(",b"exec("]
        for sig in SIGS:
            if sig.lower() in payload.lower():
                self._fire(p,dst,0,dp,proto,AttackType.PAYLOAD,ThreatLevel.CRITICAL,
                    f"Malicious payload: {sig.decode(errors='replace')}"); break
    def top_threats(self):
        return sorted([(ip,pr) for ip,pr in self.profiles.items()
                       if pr.threat_score>0],key=lambda x:-x[1].threat_score)[:10]

print("Full Threat Detector loaded!")


In [ ]:
logger   = AlertLogger(log_dir="logs")
detector = ThreatDetector(logger, ml=ml_detector)

def emit(src,dst="192.168.1.1",sp=None,dp=80,proto="TCP",flags=0x002,size=64,payload=b""):
    detector.analyze({"src_ip":src,"dst_ip":dst,
        "src_port":sp or random.randint(1024,65535),
        "dst_port":dp,"protocol":proto,"flags":flags,"size":size,"payload":payload})

print("="*60)
print("  SENTINEL IDS v3.0 -- FULL SIMULATION")
print("="*60)

print("\n[1] Port Scan -- 45.33.32.156")
for p in [21,22,23,25,53,80,110,143,443,8080,3306,5432,6379,8443,27017,9200]:
    emit("45.33.32.156",dp=p)

print("\n[2] SYN Flood -- 5.188.206.14")
for _ in range(250): emit("5.188.206.14",dp=80,flags=0x002,size=44)

print("\n[3] SSH Brute Force -- 103.21.244.0")
for _ in range(15): emit("103.21.244.0",dp=22,flags=0x018)

print("\n[4] XMAS Scan -- 198.51.100.5")
for p in [22,80,443,8080,3306]: emit("198.51.100.5",dp=p,flags=0x029)

print("\n[5] ICMP Flood -- 8.8.4.4")
for _ in range(60): emit("8.8.4.4",dp=0,proto="ICMP",flags=0)

print("\n[6] DNS Amplification -- 203.0.113.7")
for _ in range(5): emit("203.0.113.7",dp=53,proto="UDP",size=1200,flags=0)

print("\n[7] SQL Injection -- 91.108.4.1")
emit("91.108.4.1",dp=80,flags=0x018,payload=b"GET /?id=' UNION SELECT * FROM users-- HTTP/1.1")

print("\n[8] NULL Scan -- 104.21.0.1")
for p in [22,80,443]: emit("104.21.0.1",dp=p,flags=0x000)

print("\n[9] High Frequency -- 1.1.1.1")
for _ in range(120): emit("1.1.1.1",dp=80)

print("\n[10] UDP Flood -- 52.10.0.1")
for _ in range(160): emit("52.10.0.1",dp=53,proto="UDP",flags=0)

print("\n[11] Auto-blacklist trigger -- 99.99.99.99")
for _ in range(300): emit("99.99.99.99",dp=80,flags=0x002,size=44)
for p in [21,22,23,25,53,80,110,143,443,8080,3306,5432,6379,8443,27017]:
    emit("99.99.99.99",dp=p)

print("\n" + "="*60)
print("  ALL SIMULATIONS COMPLETE")
print("="*60)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
from IPython.display import display

alerts = logger.alerts
df = pd.DataFrame([a.to_dict() for a in alerts])
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["bucket"]    = df["timestamp"].dt.floor("10s")

COLORS = {"INFO":"#3498db","LOW":"#2ecc71","MEDIUM":"#f39c12",
          "HIGH":"#e74c3c","CRITICAL":"#8e44ad"}
BG = "#0d1117"; BG2 = "#161b22"; GRID = "#30363d"

fig, axes = plt.subplots(2,2,figsize=(16,10))
fig.patch.set_facecolor(BG)
fig.suptitle("SENTINEL IDS -- Threat Analytics Dashboard",
             color="white",fontsize=15,fontweight="bold",y=0.98)
for ax in axes.flat:
    ax.set_facecolor(BG2)
    ax.tick_params(colors="white")
    ax.xaxis.label.set_color("white")
    ax.yaxis.label.set_color("white")
    ax.title.set_color("white")
    for s in ax.spines.values(): s.set_color(GRID)

# Chart 1: Attack types
ax1=axes[0,0]
ac=df["attack_type"].value_counts()
bar_colors=["#e74c3c","#8e44ad","#f39c12","#3498db","#2ecc71",
            "#e67e22","#1abc9c","#e91e63","#ff5722","#607d8b","#9c27b0","#00bcd4"]
bars=ax1.barh(ac.index,ac.values,color=bar_colors[:len(ac)],edgecolor=GRID)
ax1.set_title("Attack Type Distribution",fontweight="bold")
ax1.set_xlabel("Count")
for bar,val in zip(bars,ac.values):
    ax1.text(bar.get_width()+0.05,bar.get_y()+bar.get_height()/2,
             str(val),va="center",color="white",fontsize=9)

# Chart 2: Threat level pie
ax2=axes[0,1]
lc=df["threat_level"].value_counts()
pc=[COLORS.get(l,"#666") for l in lc.index]
wedges,texts,at=ax2.pie(lc.values,labels=lc.index,colors=pc,
    autopct="%1.0f%%",startangle=90,
    textprops={"color":"white","fontsize":10},
    wedgeprops={"edgecolor":BG,"linewidth":2})
for a in at: a.set_color("white")
ax2.set_title("Threat Level Distribution",fontweight="bold")

# Chart 3: Timeline
ax3=axes[1,0]
for lvl,color in COLORS.items():
    sub=df[df["threat_level"]==lvl]
    if not sub.empty:
        tl=sub.groupby("bucket").size()
        ax3.plot(range(len(tl)),tl.values,marker="o",
                 label=lvl,color=color,linewidth=2,markersize=5)
ax3.set_title("Alert Timeline (10s buckets)",fontweight="bold")
ax3.set_xlabel("Time Bucket"); ax3.set_ylabel("Alerts")
ax3.legend(facecolor=BG2,labelcolor="white",fontsize=8)
ax3.grid(color=GRID,linestyle="--",alpha=0.5)

# Chart 4: Top offenders
ax4=axes[1,1]
top=sorted([(ip,pr.threat_score) for ip,pr in detector.profiles.items()
            if pr.threat_score>0],key=lambda x:-x[1])[:8]
ips=[t[0] for t in top]; scores=[t[1] for t in top]
bc=["#8e44ad" if detector.profiles[ip].blacklisted else "#e74c3c" for ip in ips]
ax4.bar(range(len(ips)),scores,color=bc,edgecolor=GRID)
ax4.set_xticks(range(len(ips))); ax4.set_xticklabels(ips,rotation=30,ha="right",fontsize=8)
ax4.set_title("Top Offenders by Threat Score",fontweight="bold")
ax4.set_ylabel("Threat Score")
ax4.axhline(y=100,color="#ff0000",linestyle="--",alpha=0.7)
r=mpatches.Patch(color="#8e44ad",label="Blacklisted")
o=mpatches.Patch(color="#e74c3c",label="Active")
ax4.legend(handles=[r,o],facecolor=BG2,labelcolor="white",fontsize=8)

plt.tight_layout()
plt.savefig("logs/threat_analytics.png",dpi=150,bbox_inches="tight",facecolor=BG)
plt.show()
print("Charts saved -> logs/threat_analytics.png")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import Counter

geo_counts=Counter(a.geo_info for a in alerts)
GEO_XY={"US":(0.18,0.52),"Russia":(0.65,0.65),"China":(0.74,0.52),
         "Europe":(0.50,0.65),"Asia-Pacific":(0.78,0.42),"Singapore":(0.75,0.38),
         "Australia":(0.79,0.22),"Mexico":(0.15,0.44),"Norway":(0.51,0.73),
         "Africa":(0.50,0.33)}

fig,ax=plt.subplots(figsize=(16,8))
fig.patch.set_facecolor("#0d1117")
ax.set_facecolor("#0a1628")
ax.set_xlim(0,1); ax.set_ylim(0,1); ax.set_aspect("equal")
for s in ax.spines.values(): s.set_color("#30363d")
ax.tick_params(colors="#30363d")

continents=[
    (0.05,0.35,0.24,0.36,"#111a11","Americas"),
    (0.42,0.46,0.18,0.28,"#11111a","Europe"),
    (0.44,0.18,0.14,0.24,"#1a1111","Africa"),
    (0.60,0.30,0.24,0.40,"#111a1a","Asia"),
    (0.72,0.10,0.12,0.16,"#11111a","Australia"),
]
for x,y,w,h,color,name in continents:
    ax.add_patch(patches.FancyBboxPatch((x,y),w,h,
        boxstyle="round,pad=0.01",facecolor=color,edgecolor="#2a3a4a",linewidth=1))
    ax.text(x+w/2,y+h/2,name,ha="center",va="center",
            color="#4a5a6a",fontsize=8,alpha=0.7)

maxc=max(geo_counts.values()) if geo_counts else 1
for geo,count in geo_counts.most_common():
    if geo not in GEO_XY: continue
    x,y=GEO_XY[geo]
    size=300+(count/maxc)*2500; alpha=0.4+(count/maxc)*0.5
    for ring in [3,2,1]:
        ax.scatter(x,y,s=size*ring*0.8,c="#e74c3c",alpha=alpha/(ring*2),zorder=3)
    ax.scatter(x,y,s=size,c="#ff4444",alpha=0.9,zorder=5,
               edgecolors="#ff0000",linewidths=1.5)
    ax.annotate(f"{geo}\n{count} alerts",xy=(x,y),
                xytext=(x+0.04,y+0.06),color="white",fontsize=8,fontweight="bold",
                arrowprops=dict(arrowstyle="->",color="#ff4444",lw=1),
                bbox=dict(boxstyle="round,pad=0.3",facecolor="#1a0000",
                          edgecolor="#ff4444",alpha=0.85))

ax.set_title("SENTINEL IDS -- Global Threat Origin Heatmap",
             color="white",fontsize=14,fontweight="bold",pad=15)
ax.set_xticks([]); ax.set_yticks([])
top_geo=geo_counts.most_common(1)[0][0] if geo_counts else "N/A"
ax.text(0.5,0.02,
        f"Total Alerts: {len(alerts)}  |  Unique Origins: {len(geo_counts)}  |  Top Threat Origin: {top_geo}",
        ha="center",color="#aaaaaa",fontsize=9,transform=ax.transAxes)

plt.tight_layout()
plt.savefig("logs/geo_heatmap.png",dpi=150,bbox_inches="tight",facecolor="#0d1117")
plt.show()
print("Heatmap saved -> logs/geo_heatmap.png")


In [ ]:
from fpdf import FPDF

class IDSReport(FPDF):
    def header(self):
        self.set_fill_color(13,17,23)
        self.rect(0,0,210,297,"F")
        self.set_font("Courier","B",18)
        self.set_text_color(231,76,60)
        self.cell(0,12,"SENTINEL IDS  THREAT REPORT",ln=True,align="C")
        self.set_font("Courier","",9)
        self.set_text_color(150,150,150)
        self.cell(0,6,f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",ln=True,align="C")
        self.ln(4)
    def section(self,title):
        self.set_fill_color(22,27,34)
        self.set_text_color(88,166,255)
        self.set_font("Courier","B",11)
        self.cell(0,8,f"  {title}",ln=True,fill=True)
        self.ln(2)
    def row(self,label,value,hl=False):
        self.set_font("Courier","",9)
        self.set_text_color(180,180,180)
        self.cell(60,6,f"  {label}:",0,0)
        self.set_text_color(231,76,60 if hl else 255)
        self.set_text_color(*(231,76,60) if hl else (255,255,255))
        self.cell(0,6,str(value),ln=True)

summ=logger.summary(); stats=detector.stats
pdf=IDSReport()
pdf.set_auto_page_break(auto=True,margin=15)
pdf.add_page()

pdf.section("EXECUTIVE SUMMARY")
pdf.row("Report Date",datetime.now().strftime("%Y-%m-%d"))
pdf.row("Total Packets",f"{stats['packets']:,}")
pdf.row("Total Alerts",summ["total"],hl=True)
pdf.row("Unique IPs Seen",stats["ips"])
pdf.row("IPs Auto-Blocked",stats["blocked"],hl=stats["blocked"]>0)
pdf.row("ML Model Trained",str(ml_detector.trained))
pdf.row("Uptime",str(timedelta(seconds=int(time.time()-stats["start"]))))
pdf.ln(4)

pdf.section("ATTACK BREAKDOWN")
for atk,cnt in sorted(summ["by_attack"].items(),key=lambda x:-x[1]):
    pdf.row(atk,f"{cnt} alert(s)",hl=cnt>=3)
pdf.ln(4)

pdf.section("TOP THREAT ACTORS")
for ip,cnt in summ["top_ips"]:
    pr=detector.profiles.get(ip)
    score=pr.threat_score if pr else 0
    bl=" [BLACKLISTED]" if pr and pr.blacklisted else ""
    pdf.row(f"{ip} ({geo_lookup(ip)})",f"{cnt} alerts | Score:{score:.0f}{bl}",hl=bool(bl))
pdf.ln(4)

pdf.section("BLOCKED IPs")
if detector.blocked_ips:
    for ip in detector.blocked_ips:
        pdf.row(ip,"AUTO-BLOCKED (iptables on real server)",hl=True)
else:
    pdf.row("None","No IPs reached blacklist threshold")
pdf.ln(4)

pdf.section("ALERT LOG (Last 30 Entries)")
pdf.set_font("Courier","",7)
LVL_C={"INFO":(52,152,219),"LOW":(46,204,113),"MEDIUM":(243,156,18),
        "HIGH":(231,76,60),"CRITICAL":(142,68,173)}
for a in logger.alerts[-30:]:
    c=LVL_C.get(a.threat_level,(150,150,150))
    pdf.set_text_color(*c)
    ts=a.timestamp[11:19]
    line=f"  {ts} [{a.threat_level:<8}] {a.attack_type:<22} {a.src_ip:<18} -> :{a.dst_port}"
    pdf.cell(0,5,line,ln=True)

for img_path,title in [("logs/threat_analytics.png","Analytics Charts"),
                        ("logs/geo_heatmap.png","Geographic Heatmap")]:
    if os.path.exists(img_path):
        pdf.add_page()
        pdf.section(title)
        pdf.image(img_path,x=10,w=190)

report_path="logs/sentinel_report.pdf"
pdf.output(report_path)
print(f"PDF Report generated: {report_path}")


In [ ]:
from IPython.display import display
import pandas as pd
from google.colab import files

s=detector.stats; summ=logger.summary()
uptime=timedelta(seconds=int(time.time()-s["start"]))

print("="*60)
print("  SENTINEL IDS v3.0 -- FINAL DASHBOARD")
print("="*60)
print(f"  Uptime        : {uptime}")
print(f"  Packets       : {s['packets']:,}")
print(f"  Bytes         : {s['bytes']:,}")
print(f"  Unique IPs    : {s['ips']}")
print(f"  Total Alerts  : {s['alerts']}")
print(f"  IPs Blocked   : {s['blocked']}")
print(f"  ML Trained    : {ml_detector.trained}")

print("\n  Top Threat IPs:")
for ip,pr in detector.top_threats():
    bl=" <-- BLACKLISTED" if pr.blacklisted else ""
    bar="X"*min(int(pr.threat_score/5),20)
    print(f"    {ip:<22} {pr.threat_score:<6.0f} [{bar}]{bl}")

print("\n  Attack Breakdown:")
for atk,cnt in sorted(summ["by_attack"].items(),key=lambda x:-x[1]):
    print(f"    {atk:<30} -> {cnt}")

print("\n  Auto-Blocked IPs:")
if detector.blocked_ips:
    for ip in detector.blocked_ips:
        print(f"    {ip}  (iptables rule on real Linux server)")
else:
    print("    None reached threshold yet")

print("="*60)

if logger.alerts:
    df=pd.DataFrame([a.to_dict() for a in logger.alerts])
    cols=["timestamp","src_ip","attack_type","threat_level","dst_port","geo_info","packet_count"]
    print("\nAlert Table:")
    display(df[cols])

print("\nDownloading all output files...")
for fpath in [logger.json_path,logger.csv_path,logger.suspect_path,
              "logs/threat_analytics.png","logs/geo_heatmap.png",
              "logs/sentinel_report.pdf"]:
    if os.path.exists(fpath):
        files.download(fpath)
        print(f"  Downloaded: {fpath}")
